# 05. Flower Classification using VGG16 model

The project utilizes a dataset containing images of various flower species. The dataset is divided into a training set and a test set, each labeled with the corresponding flower species. There are 102 flower species.

In [9]:
# Import libraries
from pathlib import Path
import matplotlib.pyplot as plt
import torch
import torchvision.models
from torch import nn
from torchinfo import summary
from torchvision import datasets
from torchvision import transforms
from torchvision.models import ResNet152_Weights

from src.data_loader import create_dataloaders
from src.models.vgg_16 import VGG16
from src.utils import set_device_agnostic_mode
from src.training import train_model

In [10]:
# Initialize training parameters and get device
train_dir = 'data/train'
val_dir = 'data/val'

# Get a device to use for training/inference
device = set_device_agnostic_mode()

In [11]:
# Create VGG16 CNN model

# VGG16 params and transforms
COLOR_CHANNELS = 3
BATCH_SIZE = 32
EPOCHS = 50

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1),
    transforms.RandomAffine(degrees=40, translate=None, scale=(1, 2), shear=15),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# Create training and validation data loaders
train_dataloader, val_dataloader, classes = create_dataloaders(train_dir, val_dir, train_transform, 
                                                               val_transform, BATCH_SIZE)

vgg_16_model = VGG16(COLOR_CHANNELS, len(classes))
print(summary(vgg_16_model, input_size=(BATCH_SIZE, COLOR_CHANNELS, 224, 224)))

vgg_16_model = vgg_16_model.to(device)

Layer (type:depth-idx)                   Output Shape              Param #
VGG16                                    [32, 102]                 --
├─Sequential: 1-1                        [32, 64, 224, 224]        --
│    └─Conv2d: 2-1                       [32, 64, 224, 224]        1,792
│    └─BatchNorm2d: 2-2                  [32, 64, 224, 224]        128
│    └─ReLU: 2-3                         [32, 64, 224, 224]        --
├─Sequential: 1-2                        [32, 64, 112, 112]        --
│    └─Conv2d: 2-4                       [32, 64, 224, 224]        36,928
│    └─BatchNorm2d: 2-5                  [32, 64, 224, 224]        128
│    └─ReLU: 2-6                         [32, 64, 224, 224]        --
│    └─MaxPool2d: 2-7                    [32, 64, 112, 112]        --
├─Sequential: 1-3                        [32, 128, 112, 112]       --
│    └─Conv2d: 2-8                       [32, 128, 112, 112]       73,856
│    └─BatchNorm2d: 2-9                  [32, 128, 112, 112]       256
│

In [12]:
# Train VGG16 CNN model

# Create optimizer and loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(vgg_16_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

# Train model
train_model(
    EPOCHS,
    len(classes),
    vgg_16_model,
    train_dataloader,
    val_dataloader,
    loss_fn,
    optimizer,
    device
)

Epoch 1, Train Loss 4.323634147644043, Train Accuracy 0.04727979376912117, Val Loss 4.442131519317627, Val Accuracy 0.0283203125
Epoch 2, Train Loss 4.029435157775879, Train Accuracy 0.06994818896055222, Val Loss 4.181145668029785, Val Accuracy 0.0341796875
Epoch 3, Train Loss 3.8934402465820312, Train Accuracy 0.08403497189283371, Val Loss 4.111543655395508, Val Accuracy 0.0498046875
Epoch 4, Train Loss 3.7735466957092285, Train Accuracy 0.10459844768047333, Val Loss 4.013388156890869, Val Accuracy 0.0673828125
Epoch 5, Train Loss 3.6653828620910645, Train Accuracy 0.12079015374183655, Val Loss 3.7034173011779785, Val Accuracy 0.0986328125
Epoch 6, Train Loss 3.580791473388672, Train Accuracy 0.1334196925163269, Val Loss 3.711507558822632, Val Accuracy 0.095703125
Epoch 7, Train Loss 3.4844465255737305, Train Accuracy 0.15097150206565857, Val Loss 3.691579818725586, Val Accuracy 0.103515625
Epoch 8, Train Loss 3.41715407371521, Train Accuracy 0.16305051743984222, Val Loss 3.6054217815

In [14]:
# Save VGG16 model

# Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Create model save path
MODEL_NAME = "05_flower_classification_vgg16.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=vgg_16_model.state_dict(), f=MODEL_SAVE_PATH)

Saving model to: models/05_flower_classification_vgg16.pth
